# Warehouse candle integrity audit

Notebook nghiên cứu **chỉ-đọc** cho `DWH.Fact_OHLCV`. Chọn một symbol/timeframe từ `Config.yaml`, hoặc chọn `ALL` để kiểm tra toàn bộ lựa chọn Live hiện hành. Notebook không gọi TradingView/Redis và không chạy SQL ghi dữ liệu.

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys


def locate_repository() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src' / 'dp_program').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from core_program or one of its subdirectories.')


REPOSITORY = locate_repository()
RUN_CONFIG = REPOSITORY.parent / 'run_dp' / 'Config.yaml'
sys.path.insert(0, str(REPOSITORY / 'src'))

from dp_program.configuration import load_config
from dp_program.engine.sql_connector import get_connection

assert RUN_CONFIG.is_file(), f'Production config not found: {RUN_CONFIG}'
config = load_config(RUN_CONFIG)
AVAILABLE_SYMBOLS = tuple(config['live']['symbols'])
AVAILABLE_TIMEFRAMES = tuple(config['live']['timeframes'])
print(f'Live symbols from config: {len(AVAILABLE_SYMBOLS)}')
print(f'Live timeframes from config: {len(AVAILABLE_TIMEFRAMES)}')

## Cơ chế kiểm tra

1. **Lỗi chắc chắn**: duplicate business key, timestamp có giây lẻ, OHLC vô lý, volume âm, `DateKey` không khớp `BarTime`.
2. **Gap candidate**: hai nến liên tiếp cách nhau từ hai lần timeframe trở lên. Đây chưa phải kết luận mất nến: provider có thể có maintenance hoặc phiên đóng. Notebook hiển thị riêng gap toàn lịch sử và gap trong cửa sổ gần nhất.
3. Không ép nến phải bắt đầu 00:00 UTC, vì provider có thể dùng phiên 21:00/22:00 UTC và chuyển DST.

In [ ]:
def selected_values(value: str, allowed: tuple[str, ...], label: str) -> tuple[str, ...]:
    if value == 'ALL':
        return allowed
    if value not in allowed:
        raise ValueError(f'{label} is not in the current live config: {value}')
    return (value,)


def selection_predicate(symbols: tuple[str, ...], timeframes: tuple[str, ...]) -> tuple[str, tuple[str, ...]]:
    symbol_marks = ', '.join('?' for _ in symbols)
    timeframe_marks = ', '.join('?' for _ in timeframes)
    return (
        f's.Symbol IN ({symbol_marks}) AND tf.Code IN ({timeframe_marks})',
        (*symbols, *timeframes),
    )


def ordered_cte(predicate: str) -> str:
    return f'''
WITH Ordered AS (
    SELECT s.Symbol, tf.Code, tf.Minutes, f.SymbolID, f.TimeframeID, f.BarTime, f.DateKey,
           f.[Open], f.High, f.Low, f.[Close], f.Volume, f.TickCount,
           LAG(f.BarTime) OVER (PARTITION BY f.SymbolID, f.TimeframeID ORDER BY f.BarTime) AS PreviousBarTime
    FROM DWH.Fact_OHLCV AS f
    JOIN DWH.Dim_Symbol AS s ON s.SymbolID = f.SymbolID
    JOIN DWH.Dim_Timeframe AS tf ON tf.TimeframeID = f.TimeframeID
    WHERE {predicate}
), Evaluated AS (
    SELECT *,
        CASE WHEN DATEPART(SECOND, BarTime) <> 0 THEN 1 ELSE 0 END AS TimestampSecondIssue,
        CASE WHEN PreviousBarTime IS NOT NULL
              AND DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) >= CAST(Minutes AS bigint) * 120
             THEN 1 ELSE 0 END AS GapCandidate,
        CASE WHEN PreviousBarTime IS NOT NULL
             THEN DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) END AS GapSeconds
    FROM Ordered
)
'''


def query_dicts(cursor, statement: str, parameters: tuple[object, ...]) -> list[dict[str, object]]:
    cursor.execute(statement, parameters)
    columns = [column[0] for column in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]

In [ ]:
def run_audit(
    selected_symbol: str = 'ALL',
    selected_timeframe: str = 'ALL',
    recent_window_days: int = 60,
    gap_limit: int = 40,
) -> dict[str, object]:
    if recent_window_days < 1 or gap_limit < 1:
        raise ValueError('recent_window_days and gap_limit must be positive.')
    symbols = selected_values(selected_symbol, AVAILABLE_SYMBOLS, 'symbol')
    timeframes = selected_values(selected_timeframe, AVAILABLE_TIMEFRAMES, 'timeframe')
    predicate, base_parameters = selection_predicate(symbols, timeframes)
    cte = ordered_cte(predicate)

    summary_sql = cte + '''
SELECT Symbol, Code, Minutes, COUNT_BIG(*) AS [Rows], MIN(BarTime) AS FirstBarTime, MAX(BarTime) AS LastBarTime,
       SUM(CASE WHEN TimestampSecondIssue = 1 THEN 1 ELSE 0 END) AS TimestampSecondIssues,
       SUM(CASE WHEN [Open] <= 0 OR High <= 0 OR Low <= 0 OR [Close] <= 0
                     OR High < [Open] OR High < [Close] OR Low > [Open] OR Low > [Close] OR High < Low
                THEN 1 ELSE 0 END) AS OhlcIssues,
       SUM(CASE WHEN Volume < 0 THEN 1 ELSE 0 END) AS NegativeVolumeIssues,
       SUM(CASE WHEN DateKey <> CONVERT(int, CONVERT(char(8), BarTime, 112)) THEN 1 ELSE 0 END) AS DateKeyIssues,
       SUM(CASE WHEN TickCount IS NOT NULL AND TickCount <> 1 THEN 1 ELSE 0 END) AS NonDefaultTickCountRows,
       SUM(CASE WHEN GapCandidate = 1 THEN 1 ELSE 0 END) AS AllHistoryGapCandidates,
       MAX(GapSeconds) AS LargestGapSeconds
FROM Evaluated
GROUP BY Symbol, Code, Minutes
ORDER BY Symbol, Minutes;
'''
    duplicate_sql = f'''
WITH Duplicates AS (
    SELECT s.Symbol, tf.Code, tf.Minutes
    FROM DWH.Fact_OHLCV AS f
    JOIN DWH.Dim_Symbol AS s ON s.SymbolID = f.SymbolID
    JOIN DWH.Dim_Timeframe AS tf ON tf.TimeframeID = f.TimeframeID
    WHERE {predicate}
    GROUP BY s.Symbol, tf.Code, tf.Minutes, f.BarTime
    HAVING COUNT_BIG(*) > 1
)
SELECT Symbol, Code, COUNT_BIG(*) AS DuplicateBusinessKeys
FROM Duplicates
GROUP BY Symbol, Code
ORDER BY Symbol, MIN(Minutes);
'''
    gap_sql = cte + '''
SELECT TOP (?) Symbol, Code, Minutes, PreviousBarTime, BarTime,
       DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) AS GapSeconds,
       DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) / (CAST(Minutes AS bigint) * 60) - 1 AS CandidateMissingBars
FROM Evaluated
WHERE GapCandidate = 1
ORDER BY GapSeconds DESC, Symbol, BarTime DESC;
'''
    recent_gap_sql = cte + '''
SELECT TOP (?) Symbol, Code, Minutes, PreviousBarTime, BarTime,
       DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) AS GapSeconds,
       DATEDIFF_BIG(SECOND, PreviousBarTime, BarTime) / (CAST(Minutes AS bigint) * 60) - 1 AS CandidateMissingBars
FROM Evaluated
WHERE GapCandidate = 1 AND PreviousBarTime >= ? AND BarTime <= ?
ORDER BY GapSeconds DESC, Symbol, BarTime DESC;
'''
    recent_summary_sql = cte + '''
SELECT Symbol, Code, Minutes, COUNT_BIG(*) AS RecentGapCandidates,
       MIN(PreviousBarTime) AS FirstPreviousBarTime, MAX(BarTime) AS LastBarTime,
       MAX(GapSeconds) AS LargestGapSeconds
FROM Evaluated
WHERE GapCandidate = 1 AND PreviousBarTime >= ? AND BarTime <= ?
GROUP BY Symbol, Code, Minutes
ORDER BY Symbol, Minutes;
'''

    window_end = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
    window_start = window_end - timedelta(days=recent_window_days)
    connection = get_connection(config)
    try:
        cursor = connection.cursor()
        summary = query_dicts(cursor, summary_sql, base_parameters)
        duplicates = query_dicts(cursor, duplicate_sql, base_parameters)
        all_history_gaps = query_dicts(cursor, gap_sql, (*base_parameters, gap_limit))
        recent_gaps = query_dicts(cursor, recent_gap_sql, (*base_parameters, gap_limit, window_start, window_end))
        recent_gap_summary = query_dicts(cursor, recent_summary_sql, (*base_parameters, window_start, window_end))
    finally:
        connection.close()

    duplicate_map = {(item['Symbol'], item['Code']): int(item['DuplicateBusinessKeys']) for item in duplicates}
    hard_columns = ('DuplicateBusinessKeys', 'TimestampSecondIssues', 'OhlcIssues', 'NegativeVolumeIssues', 'DateKeyIssues')
    hard_issue_pairs = []
    for item in summary:
        item['DuplicateBusinessKeys'] = duplicate_map.get((item['Symbol'], item['Code']), 0)
        if any(int(item[column] or 0) for column in hard_columns):
            hard_issue_pairs.append(f"{item['Symbol']}/{item['Code']}")
    return {
        'captured_at_utc': datetime.now(timezone.utc).isoformat(),
        'symbols': symbols,
        'timeframes': timeframes,
        'recent_window_start_utc': window_start.isoformat(sep=' '),
        'recent_window_end_utc': window_end.isoformat(sep=' '),
        'hard_issue_pairs': hard_issue_pairs,
        'summary': summary,
        'all_history_gap_candidates': all_history_gaps,
        'recent_gap_candidates': recent_gaps,
        'recent_gap_summary': recent_gap_summary,
    }

In [ ]:
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError as exc:
    raise RuntimeError('This interactive notebook needs ipywidgets and pandas installed in its Jupyter kernel.') from exc

symbol_control = widgets.Dropdown(options=('ALL', *AVAILABLE_SYMBOLS), value='ALL', description='Symbol:')
timeframe_control = widgets.Dropdown(options=('ALL', *AVAILABLE_TIMEFRAMES), value='ALL', description='Timeframe:')
window_control = widgets.BoundedIntText(value=60, min=1, max=3650, description='Recent days:')
gap_limit_control = widgets.BoundedIntText(value=40, min=1, max=1000, description='Gap rows:')
run_button = widgets.Button(description='Run read-only audit', button_style='primary', icon='search')
output = widgets.Output()


def show_report(report: dict[str, object]) -> None:
    print(f"Captured UTC: {report['captured_at_utc']}")
    print('Hard issue pairs:', report['hard_issue_pairs'] or 'none')
    print(f"Recent window: {report['recent_window_start_utc']} -> {report['recent_window_end_utc']}")
    display(pd.DataFrame(report['summary']))
    print('Largest all-history gap candidates:')
    display(pd.DataFrame(report['all_history_gap_candidates']))
    print('Recent gap candidate summary:')
    display(pd.DataFrame(report['recent_gap_summary']))
    print('Recent gap candidate details:')
    display(pd.DataFrame(report['recent_gap_candidates']))


def on_run(_button) -> None:
    with output:
        clear_output(wait=True)
        report = run_audit(
            symbol_control.value, timeframe_control.value,
            window_control.value, gap_limit_control.value,
        )
        show_report(report)


run_button.on_click(on_run)
display(widgets.VBox([
    widgets.HBox([symbol_control, timeframe_control]),
    widgets.HBox([window_control, gap_limit_control, run_button]),
    output,
]))

## Cách đọc kết quả

- `Hard issue pairs = none`: không thấy vi phạm cấu trúc chắc chắn trong phạm vi đã chọn.
- Gap toàn lịch sử cho biết tính liên tục của kho dữ liệu hiện hữu; gap rất lớn thường phản ánh lịch sử được nạp từng đợt, không tự chứng minh hỏng pipeline hiện tại.
- Gap gần đây cần được đọc theo mẫu thời gian. Gap lặp cùng giờ/ngày thường là maintenance/session của provider; gap bất thường cần đối chiếu TradingView trước khi backfill.
- Notebook chỉ hiển thị bằng chứng. Nó không tự backfill, sửa Fact hay publish Redis.